In [ ]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.cm as cm

In [ ]:
folder_path = r"/storage/alplakes_test/lucerne_100m_2025"
catalogue_folder = os.path.join(folder_path, "outputs_swirl", "eddy_catalogues_final")

output_folder = os.path.join(folder_path, "outputs_swirl", "specific_eddy_plots")
os.makedirs(output_folder, exist_ok=True)

# Import catalogues

In [ ]:
lake_csv_path = os.path.join(catalogue_folder, "lake_characteristics.csv")
df_lake = pd.read_csv(lake_csv_path)
df_lake = df_lake.set_index('id', drop=False)
df_lake['date'] = pd.to_datetime(df_lake['date'])

In [ ]:
lvl0_csv_path = os.path.join(catalogue_folder, "lvl0.csv")
df_lvl0 = pd.read_csv(lvl0_csv_path)
df_lvl0 = df_lvl0.set_index('id', drop=False)
df_lvl0['date'] = pd.to_datetime(df_lvl0['date'])

In [ ]:
lvl1_csv_path = os.path.join(catalogue_folder, "lvl1.csv")
df_lvl1 = pd.read_csv(lvl1_csv_path)
df_lvl1 = df_lvl1.set_index('id', drop=False)
df_lvl1['date'] = pd.to_datetime(df_lvl1['date'])

In [ ]:
lvl2_csv_path = os.path.join(catalogue_folder, "lvl2.csv")
df_lvl2 = pd.read_csv(lvl2_csv_path)
df_lvl2 = df_lvl2.set_index('id', drop=False)

# Import lake mask

In [ ]:
lake_mask = np.load(os.path.join(folder_path, "grid", "mask_lake.npy"))
Nx = lake_mask.shape[2]
Ny = lake_mask.shape[1]

# Plot from lvl0 catalogue

In [ ]:
import ast

In [ ]:
def to_array(x):
    if isinstance(x, str):
        return np.array(ast.literal_eval(x))
    return np.array(x)

In [ ]:
def plot_eddy_lvl0(lvl0_row, Nx, Ny, color='Blues', alpha=1):
    i = to_array(lvl0_row['i_eddy_cells']).astype(int)
    j = to_array(lvl0_row['j_eddy_cells']).astype(int)

    img = np.zeros((Ny, Nx))
    img[j, i] = 1
    img[img == 0] = np.nan

    plt.imshow(
        img,
        cmap=color,
        origin='lower',
        vmin=0,
        vmax=1,
        alpha=alpha,
    )

In [ ]:
from matplotlib.colors import ListedColormap

lake_cmap = ListedColormap(["white","#cfe8ff"])
cmap_blue = ListedColormap(["#135a8a"])
cmap_red = ListedColormap(["#d9091d"])
cmap_yellow = ListedColormap(["#f2e602"])

In [ ]:
plt.imshow(
    lake_mask[0],
    cmap=lake_cmap,
    origin='lower',
    alpha=0.5
)

plot_eddy_lvl0(df_lvl0.iloc[0], Nx, Ny, cmap_blue)
plot_eddy_lvl0(df_lvl0.iloc[1], Nx, Ny, cmap_red)
plot_eddy_lvl0(df_lvl0.iloc[2], Nx, Ny, cmap_yellow)

# Plot eddy from lvl1 catalogue

In [ ]:
def plot_lvl1_eddy(df_lvl0, id_lvl1, df_lvl1, Nx, Ny, color=cmap_yellow, alpha=0.5):
    ids_lvl0 = to_array(df_lvl1.iloc[id_lvl1]['id_lvl0']).astype(int)
    for id_lvl0 in ids_lvl0:
        plot_eddy_lvl0(df_lvl0.iloc[id_lvl0], Nx, Ny, color=color, alpha=alpha)

In [ ]:
id_lvl1 = 5
plot_lvl1_eddy(df_lvl0, id_lvl1, df_lvl1, Nx, Ny, color=cmap_yellow, alpha=0.5)

# Plot eddy from lvl2 catalogue

In [ ]:
id_lvl2 = 99355

In [ ]:
lvl2_row = df_lvl2.iloc[id_lvl2]
ids_lvl1 = to_array(lvl2_row['id_lvl1']).astype(int)

In [ ]:
plt.figure(figsize=(14,10))

plt.imshow(lake_mask[0], cmap=lake_cmap, origin='lower', alpha=0.5)

for id_lvl1 in ids_lvl1:
    rand_color = cm.tab20(np.random.rand())
    cmap_random = ListedColormap([rand_color])
    plot_lvl1_eddy(df_lvl0, id_lvl1, df_lvl1, Nx, Ny, color=cmap_random, alpha=0.5)

for id_lvl1 in ids_lvl1:
    row_lvl1 = df_lvl1.iloc[id_lvl1]
    plt.scatter(row_lvl1['xc_mean'], row_lvl1['yc_mean'], color='r', marker='x', s=0.2)

plt.text(0.98, 0.98, f'id_lvl2 = {id_lvl2}', transform=plt.gca().transAxes, ha='right', va='top')

#plt.savefig(os.path.join(output_folder, f"trajectory_{id_lvl2}.png"))

In [ ]:
plt.imshow(lake_mask[0], cmap=lake_cmap, origin='lower', alpha=0.5)
for id_lvl1 in ids_lvl1:
    row_lvl1 = df_lvl1.iloc[id_lvl1]
    plt.scatter(row_lvl1['xc_mean'], row_lvl1['yc_mean'], color='r', marker='x', s=0.2)